In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

True

In [ ]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression such as 25 * (10 + 2)."""
    import ast
    import operator

    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.USub: operator.neg,
        ast.UAdd: operator.pos,
        ast.Mod: operator.mod,
    }

    def evaluate(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value

        if isinstance(node, ast.BinOp) and type(node.op) in operators:
            return operators[type(node.op)](
                evaluate(node.left), evaluate(node.right)
            )

        if isinstance(node, ast.UnaryOp) and type(node.op) in operators:
            return operators[type(node.op)](evaluate(node.operand))

        raise ValueError("Unsupported arithmetic expression")

    tree = ast.parse(expression, mode="eval")
    return str(evaluate(tree.body))


@tool
def analyze_text(text: str) -> str:
    """Count the words and characters in a piece of text."""
    words = text.split()
    return (
        f"Word count: {len(words)}\n"
        f"Character count: {len(text)}"
    )


tools = [calculator, analyze_text]

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
)

llm_with_tools = llm.bind_tools(tools)

In [ ]:
tool_map = {tool.name: tool for tool in tools}

def run_agent(question: str, max_steps: int = 5):
    messages = [HumanMessage(content=question)]

    for _ in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # If no tools were requested, return the final answer
        if not response.tool_calls:
            return response.text

        # Execute requested tools
        for tool_call in response.tool_calls:
            name = tool_call["name"]
            args = tool_call["args"]

            print(f"Calling tool: {name}({args})")

            try:
                result = tool_map[name].invoke(args)
            except Exception as e:
                result = f"Tool error: {e}"

            print(f"📋 Tool result: {result}")

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"],
                )
            )

    return "Stopped: maximum agent steps reached."

In [10]:
#run_agent("Calculate 125 * 48 and explain the result.")
run_agent("Analyze this text: LangChain makes it easier to build AI agents.")

🔧 Calling tool: analyze_text({'text': 'LangChain makes it easier to build AI agents.'})
📋 Tool result: Word count: 8
Character count: 45


'The text "LangChain makes it easier to build AI agents." contains 8 words and 45 characters. \n\n**Linguistic Analysis:**\n*   **Subject:** "LangChain" (a specific software framework).\n*   **Verb:** "makes" (indicates a function or result).\n*   **Object/Complement:** "it easier to build AI agents" (describes the benefit or utility of the subject).\n*   **Tone:** Informative, positive, and concise.\n*   **Purpose:** To highlight the value proposition of LangChain in the context of AI development.'